In [1]:
import os
import glob
import pickle
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.decomposition import NMF
from sklearn.ensemble import RandomForestClassifier
import warnings

In [2]:
# 1. On ne prend QUE les configurations où les Stopwords sont supprimés (S1)
chemin_dossier_pkl = 'vectorisation-du-texte/output/'
# L'astuce est ici : on filtre avec '*_S1_*'
fichiers_pkl_propres = glob.glob(os.path.join(chemin_dossier_pkl, '*_S1_*_FINAL.pkl'))

print(f"{len(fichiers_pkl_propres)} fichiers SANS stopwords trouvés.")

12 fichiers SANS stopwords trouvés.


In [3]:
# 2. Les hyperparamètres de la NMF à tester (demandés par le sujet)
parametres_nmf = [
    {'n_components': 4, 'beta_loss': 'frobenius', 'solver': 'cd'},
    {'n_components': 5, 'beta_loss': 'frobenius', 'solver': 'cd'},
    {'n_components': 4, 'beta_loss': 'kullback-leibler', 'solver': 'mu'}, # KL nécessite le solver 'mu'
    {'n_components': 5, 'beta_loss': 'kullback-leibler', 'solver': 'mu'}
]

resultats_themes = []

# 3. La boucle d'évaluation
for chemin_fichier in fichiers_pkl_propres:
    nom_fichier = os.path.basename(chemin_fichier).replace('_FINAL.pkl', '')
    
    # Chargement
    with open(chemin_fichier, 'rb') as f:
        data = pickle.load(f)
        
    X = data['X_normalized']
    y = data['target']
    
    # Pour chaque fichier, on teste les paramètres NMF
    for params in parametres_nmf:
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                
                # Étape A : Extraire les thèmes avec NMF
                nmf = NMF(n_components=params['n_components'], 
                          beta_loss=params['beta_loss'], 
                          solver=params['solver'], 
                          init='nndsvda', # Init recommandée pour KL
                          random_state=42, 
                          max_iter=500)
                
                # W est la nouvelle matrice : chaque avis est représenté par ses thèmes
                W = nmf.fit_transform(X) 
                
                # Étape B : Évaluer ces thèmes avec le Random Forest !
                # On sépare notre nouvelle matrice W en train/test
                W_train, W_test, y_train, y_test = train_test_split(W, y, test_size=0.2, random_state=42)
                
                rf = RandomForestClassifier(n_estimators=100, random_state=42)
                rf.fit(W_train, y_train)
                
                score_rf = rf.score(W_test, y_test)
                
                resultats_themes.append({
                    'Configuration Texte': nom_fichier,
                    'Nb Thèmes': params['n_components'],
                    'Fonction Perte NMF': params['beta_loss'],
                    'Score Random Forest (%)': round(score_rf * 100, 2)
                })
        except Exception as e:
            # Certains algorithmes mathématiques peuvent échouer selon les données
            continue

In [4]:
# 4. Affichage du classement final
print("CLASSEMENT DES CONFIGURATIONS THÉMATIQUES (NMF + RANDOM FOREST)")
df_themes = pd.DataFrame(resultats_themes)
df_themes = df_themes.sort_values(by='Score Random Forest (%)', ascending=False).reset_index(drop=True)
df_themes.index = df_themes.index + 1 

display(df_themes.head(10)) # On affiche le Top 10

meilleur_theme = df_themes.iloc[0]
print("\n" + "*"*80)
print(f"🎯 CONCLUSION MISSION 2 : ")
print(f"Pour extraire les thèmes, utilisez le fichier '{meilleur_theme['Configuration Texte']}'")
print(f"avec une NMF configurée avec {meilleur_theme['Nb Thèmes']} thèmes et la perte '{meilleur_theme['Fonction Perte NMF']}'.")
print("*"*80)

CLASSEMENT DES CONFIGURATIONS THÉMATIQUES (NMF + RANDOM FOREST)


,Configuration Texte,Nb Thèmes,Fonction Perte NMF,Score Random Forest (%)
1,config_L0_S1_LEM1_NG1,5,kullback-leibler,78.11
2,config_L1_S1_LEM0_NG1,5,kullback-leibler,77.57
3,config_L1_S1_LEM1_NG3,5,kullback-leibler,75.68
4,config_L0_S1_LEM1_NG3,5,kullback-leibler,75.41
5,config_L0_S1_LEM1_NG2,5,kullback-leibler,75.41
6,config_L1_S1_LEM0_NG3,5,kullback-leibler,75.41
7,config_L1_S1_LEM1_NG1,5,kullback-leibler,75.41
8,config_L0_S1_LEM1_NG2,5,frobenius,74.86
9,config_L1_S1_LEM1_NG3,5,frobenius,74.59
10,config_L0_S1_LEM1_NG3,5,frobenius,74.32



********************************************************************************
🎯 CONCLUSION MISSION 2 : 
Pour extraire les thèmes, utilisez le fichier 'config_L0_S1_LEM1_NG1'
avec une NMF configurée avec 5 thèmes et la perte 'kullback-leibler'.
********************************************************************************


In [6]:
#AFFICHAGE DES MOTS-CLÉS DU MEILLEUR MODÈLE ---

print("\n" + "="*80)
print("🔍 DÉCOUVERTE DES THÈMES DU MEILLEUR MODÈLE")
print("="*80)

# 1. On récupère le chemin du fichier gagnant
chemin_gagnant = os.path.join(chemin_dossier_pkl, meilleur_theme['Configuration Texte'] + '_FINAL.pkl')

# 2. On charge ses données (pour avoir X et surtout les mots du vocabulaire)
with open(chemin_gagnant, 'rb') as f:
    donnees_gagnantes = pickle.load(f)
    
X_gagnant = donnees_gagnantes['X_normalized']
mots_vocabulaire = donnees_gagnantes['feature_names']

# 3. On recrée le solveur en fonction de la perte gagnante
solveur_gagnant = 'mu' if meilleur_theme['Fonction Perte NMF'] == 'kullback-leibler' else 'cd'

# 4. On ré-entraîne juste la NMF gagnante (c'est très rapide pour un seul modèle)
nmf_gagnante = NMF(
    n_components=meilleur_theme['Nb Thèmes'], 
    beta_loss=meilleur_theme['Fonction Perte NMF'], 
    solver=solveur_gagnant,
    init='nndsvda',
    random_state=42, 
    max_iter=500
)

nmf_gagnante.fit(X_gagnant)

# 5. On affiche le Top 10 des mots pour chaque thème
def afficher_top_mots(model, feature_names, n_top_words=10):
    for topic_idx, topic in enumerate(model.components_):
        # On trie les poids pour trouver les mots les plus importants
        top_features_ind = topic.argsort()[:-n_top_words - 1:-1]
        top_features = [feature_names[i] for i in top_features_ind]
        print(f"📌 Thème {topic_idx + 1} : {', '.join(top_features)}")

afficher_top_mots(nmf_gagnante, mots_vocabulaire, 10)
print("="*80)


🔍 DÉCOUVERTE DES THÈMES DU MEILLEUR MODÈLE
📌 Thème 1 : le, trop, petit, être, très, peu, bracelet, dommage, plus, fragile
📌 Thème 2 : recevoir, avoir, je, ne, jamais, non, article, être, toujours, ce
📌 Thème 3 : qualité, bon, prix, produit, mauvais, rapport, très, correspondre, recommander, le
📌 Thème 4 : boucle, oreille, lui, de, jolie, d, très, Boucles, bel, photo
📌 Thème 5 : très, joli, beau, cadeau, bel, bien, conforme, recommander, bracelet, parfaire
